The [Sator square](https://en.wikipedia.org/wiki/Sator_Square) is one of the most famous puzzles from the ancient world—a five-word grid found scratched into walls at Pompeii (CIL IV 8623), Dura-Europos, and sites across the Roman Empire:

```
R O T A S
O P E R A
T E N E T
A R E P O
S A T O R
```

Each row reads the same as a column, and the whole grid is a palindrome—it reads the same top-to-bottom and bottom-to-top. But can we find *other* Latin word squares like it? How many such grids can be built from attested Latin vocabulary?

To find out, we take a comprehensive list of Latin word forms from the [LatinCy Words](https://github.com/diyclassics/latincy-words) dataset—over 919,000 forms compiled from Wiktionary entries and Universal Dependencies treebanks—and search for all four- and five-letter "magic word squares": grids where every row and column is a valid Latin word, and the grid reads the same forward and backward.

**NB**: This is an updated version of a [notebook](https://github.com/diyclassics/ll-experiments) I originally wrote in September 2019, inspired by a [tweet from Sarah Bond](https://twitter.com/SarahEBond/status/1172256207408697344) about the Sator square. The original used CLTK and the Latin Library corpus; the code below uses the LatinCy Words form list.

## Setup

In [ ]:
from random import sample, seed
from urllib.request import urlopen

seed(42)

## Loading the word list

We use the comprehensive Latin word forms list from the [LatinCy Words](https://github.com/diyclassics/latincy-words) dataset. This file contains over 919,000 attested Latin forms compiled from Wiktionary and Universal Dependencies treebanks. We extract those with exactly four or five letters.

In [ ]:
url = "https://raw.githubusercontent.com/diyclassics/latincy-words/main/data/final/latin-words.txt"
response = urlopen(url)
all_words = set(line.decode('utf-8').strip() for line in response)

words_four = set(w for w in all_words if len(w) == 4)
words_five = set(w for w in all_words if len(w) == 5)

print(f'{len(all_words):,} total forms loaded')
print(f'{len(words_four):,} four-letter words')
print(f'{len(words_five):,} five-letter words')

## What is a magic word square?

A magic word square is a grid of words where:
1. Every row is a valid word
2. Every column reads the same as the corresponding row
3. The grid is a palindrome—it reads the same top-to-bottom and bottom-to-top

This means the first row is the reverse of the last row, the second row is the reverse of the second-to-last, and so on. For a 4×4 square, if the rows are `w1, w2, w3, w4`, then `w4 = reverse(w1)` and `w3 = reverse(w2)`. The column constraint further requires that specific letters line up.

The first step is to find words whose reversal is also a valid Latin word.

In [ ]:
rev = lambda w: w[::-1]

four_valid = [w for w in words_four if rev(w) in words_four]
five_valid = [w for w in words_five if rev(w) in words_five]

print(f'{len(four_valid)} four-letter words whose reversal is also a valid word')
print(f'{len(five_valid)} five-letter words whose reversal is also a valid word')

## Finding four-letter squares

For a 4×4 square, we need two words `w1` and `w2` such that:
- `w1`, `w2`, `reverse(w2)`, and `reverse(w1)` are all valid words
- The columns also spell valid words, which means `w2[0] == w1[1]` and `w2[3] == w1[2]`

In [ ]:
four_squares = []

for w1 in four_valid:
    for w2 in four_valid:
        if w2[0] == w1[1] and w2[-1] == w1[-2]:
            four_squares.append([w1, w2, rev(w2), rev(w1)])

print(f'{len(four_squares)} four-letter magic squares found')

In [ ]:
def print_square(square):
    """Display a word square as a formatted grid."""
    for word in square:
        print(' '.join(word.upper()))
    print()

# Show a random example
print('Random four-letter square:\n')
print_square(sample(four_squares, 1)[0])

## Finding five-letter squares

Five-letter squares are harder. We now need three words `w1`, `w2`, `w3` where `w3` is its own palindrome (it occupies the middle row, which must equal its own reversal). The column constraints are tighter: `w2[0] == w1[1]`, `w2[4] == w1[3]`, `w3[0] == w1[2]`, `w3[1] == w2[2]`.

In [ ]:
five_squares = []

for w1 in five_valid:
    for w2 in five_valid:
        if w2[0] == w1[1] and w2[-1] == w1[-2]:
            for w3 in five_valid:
                if w3[0] == w1[2] and w3[1] == w2[2] and w3 == rev(w3):
                    five_squares.append([w1, w2, w3, rev(w2), rev(w1)])

print(f'{len(five_squares)} five-letter magic squares found')

In [ ]:
# Show all five-letter squares
for i, square in enumerate(five_squares, 1):
    print(f'Square {i}:')
    print_square(square)

## Checking the famous squares

Does the ROMA/OLIM/MILO/AMOR square from Sarah Bond's tweet appear in our results?

In [ ]:
bond_square = ['roma', 'olim', 'milo', 'amor']

if bond_square in four_squares:
    print('ROMA square found!')
    print_square(bond_square)
else:
    missing = [w for w in bond_square if w not in words_four]
    print(f'ROMA square not found. Missing words: {missing}')

What about the Sator square itself?

In [ ]:
sator_square = ['rotas', 'opera', 'tenet', 'arepo', 'sator']

if sator_square in five_squares:
    print('Sator square found!')
    print_square(sator_square)
else:
    missing = [w for w in sator_square if w not in words_five]
    print(f'Sator square not found. Missing words: {missing}')
    print()
    print('This is the puzzle\'s most famous feature: "arepo" is not an attested')
    print('Latin word. It may be a proper name, a borrowing, or simply a coinage')
    print('invented to complete the square.')

## All four-letter squares

For reference, here are all the four-letter squares found, displayed as word lists.

In [ ]:
for square in four_squares:
    print(square)

<hr>

<p><em>Originally written September 13, 2019. Updated April 2026 to use the LatinCy Words dataset.</em></p>